# 5-dars: Model Baholash (Model Evaluation)

## 📚 Dars Maqsadi
Ushbu darsda biz machine learning modellarini qanday to'g'ri baholashni o'rganamiz:
- **Train/Test Split** - Ma'lumotlarni to'g'ri bo'lish
- **Cross-Validation** - Ishonchli baholash
- **Overfitting/Underfitting** - Muammolarni aniqlash
- **Classification Metrics** - Accuracy, Precision, Recall, F1
- **ROC Curve va AUC** - Threshold tuning

---

In [ ]:
# Kerakli kutubxonalarni import qilish
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Model selection
from sklearn.model_selection import (
    train_test_split, cross_val_score, 
    KFold, StratifiedKFold, cross_validate
)

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score, auc,
    mean_absolute_error, mean_squared_error, r2_score
)

# Models
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification, load_breast_cancer, load_diabetes

import warnings
warnings.filterwarnings('ignore')

# Visualization sozlamalari
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Barcha kutubxonalar muvaffaqiyatli yuklandi!")

---

# 1️⃣ Train/Test Split

## 📖 Nazariya

**Nega kerak?**
- Model **o'rganish** uchun train data
- Model **baholash** uchun test data
- Test data - model hech qachon ko'rmagan!

### Asosiy Tamoyil:
**HECH QACHON test data bilan train qilmaslik!**

### Train/Test/Validation Split

```
Butun Dataset (100%)
    |
    |----> Train Set (60-80%)
    |        - Model o'rganish
    |
    |----> Validation Set (10-20%)
    |        - Hyperparameter tuning
    |        - Model selection
    |
    |----> Test Set (10-20%)
             - Final evaluation
             - Faqat bir marta!
```

### Optimal Split Ratios:

| Dataset Size | Split Ratio | Test Size |
|--------------|-------------|----------|
| Kichik (< 1000) | 70/30 | 30% |
| O'rtacha (1k-10k) | 80/20 | 20% |
| Katta (> 10k) | 90/10 | 10% |
| Juda katta (> 100k) | 95/5 | 5% |

---

## 🎨 Train/Test Split: Vizual Demo

In [ ]:
# Synthetic data yaratish
np.random.seed(42)
X, y = make_classification(n_samples=200, n_features=2, n_informative=2,
                           n_redundant=0, n_clusters_per_class=1,
                           flip_y=0.1, random_state=42)

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original Dataset
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=60, alpha=0.7, edgecolors='k')
axes[0].set_title('Original Dataset (200 samples)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].grid(True, alpha=0.3)

# Train Set
axes[1].scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='viridis',
                s=60, alpha=0.7, edgecolors='k')
axes[1].set_title(f'Train Set ({len(X_train)} samples, 80%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].grid(True, alpha=0.3)

# Test Set
axes[2].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='viridis',
                s=60, alpha=0.7, edgecolors='k')
axes[2].set_title(f'Test Set ({len(X_test)} samples, 20%)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Feature 1')
axes[2].set_ylabel('Feature 2')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("TRAIN/TEST SPLIT NATIJALAR")
print("="*70)
print(f"Original dataset: {len(X)} samples")
print(f"Train set: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test set: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nClass distribution (original): {np.bincount(y)}")
print(f"Class distribution (train): {np.bincount(y_train)}")
print(f"Class distribution (test): {np.bincount(y_test)}")
print("\n💡 stratify=y ishlatildi - class balance saqlanadi!")
print("="*70)

## 💻 Train/Test Split: Amaliy Misol

In [ ]:
# Model o'rgatish va baholash
# Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # FAQAT transform, fit emas!

# Model: Logistic Regression
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Evaluation
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Difference:     {abs(train_acc - test_acc):.4f}")

if abs(train_acc - test_acc) < 0.05:
    print("\n✅ Good fit: Train va Test accuracy yaqin!")
elif train_acc > test_acc + 0.1:
    print("\n⚠️ Overfitting: Train accuracy juda yuqori!")
else:
    print("\n⚠️ Underfitting: Ikkala accuracy ham past!")
print("="*60)

### ⚠️ Muhim: Test Data Leakage

**Nima qilmaslik kerak:**

```python
# ❌ NOTO'G'RI: Butun data'ni standardize qilish
X_scaled = scaler.fit_transform(X)  # Test data ham ko'rdi!
X_train, X_test = train_test_split(X_scaled, ...)

# ❌ NOTO'G'RI: Test data bilan fit qilish
scaler.fit(X_test)
```

**To'g'ri yo'l:**

```python
# ✅ TO'G'RI: Avval split, keyin fit
X_train, X_test = train_test_split(X, ...)
X_train_scaled = scaler.fit_transform(X_train)  # Faqat train
X_test_scaled = scaler.transform(X_test)  # Faqat transform
```

---

# 2️⃣ Cross-Validation

## 📖 Nazariya

**Muammo**: Bir marta train/test split - chance'ga bog'liq!

**Yechim**: **Cross-Validation** - Ko'p marta test qilish va o'rtachani olish.

### K-Fold Cross-Validation

```
Iteration 1: [Test][Train][Train][Train][Train]
Iteration 2: [Train][Test][Train][Train][Train]
Iteration 3: [Train][Train][Test][Train][Train]
Iteration 4: [Train][Train][Train][Test][Train]
Iteration 5: [Train][Train][Train][Train][Test]

Final Score = Average of 5 test scores
```

### K-Fold qadamlari:
1. Ma'lumotlarni **K ta** teng qismga bo'lish
2. Har bir **fold** navbatda **test set** bo'ladi
3. Qolgan **K-1** fold **train set**
4. K ta score'ning **o'rtachasi** = final score

### Cross-Validation turlari:

| Tur | Tavsif | Qachon |
|-----|--------|--------|
| **K-Fold** | K=5 yoki 10 | Standard case |
| **Stratified K-Fold** | Class balance saqlanadi | Imbalanced data |
| **Leave-One-Out (LOO)** | K=n (har bir sample) | Kichik dataset |
| **Time Series Split** | Chronological order | Time series |

---

## 🎨 K-Fold Cross-Validation: Vizual Demo

In [ ]:
# K-Fold visualization
from sklearn.model_selection import KFold

# Sample data (20 samples)
n_samples = 20
X_cv = np.arange(n_samples).reshape(-1, 1)
y_cv = np.random.randint(0, 2, n_samples)

# K-Fold (K=5)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Visualization
fig, axes = plt.subplots(5, 1, figsize=(14, 10))

for fold_idx, (train_idx, test_idx) in enumerate(kfold.split(X_cv)):
    # Create array for visualization
    fold_viz = np.full(n_samples, 0)  # 0 = train
    fold_viz[test_idx] = 1  # 1 = test
    
    # Plot
    colors = ['steelblue' if x == 0 else 'orange' for x in fold_viz]
    axes[fold_idx].bar(range(n_samples), np.ones(n_samples), color=colors, edgecolor='black')
    axes[fold_idx].set_title(f'Fold {fold_idx + 1}: Train={len(train_idx)}, Test={len(test_idx)}',
                             fontsize=12, fontweight='bold')
    axes[fold_idx].set_xlim(-0.5, n_samples - 0.5)
    axes[fold_idx].set_yticks([])
    axes[fold_idx].set_xlabel('Sample Index')
    
    # Legend (faqat birinchi fold uchun)
    if fold_idx == 0:
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='steelblue', edgecolor='black', label='Train Set'),
            Patch(facecolor='orange', edgecolor='black', label='Test Set')
        ]
        axes[fold_idx].legend(handles=legend_elements, loc='upper right', fontsize=10)

plt.suptitle('5-Fold Cross-Validation Visualization', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Har bir fold navbatda test set bo'ladi!")
print("   Final score = 5 ta test score'ning o'rtachasi")

## 💻 Cross-Validation: Amaliy Misol

In [ ]:
# Breast Cancer dataset
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target

# Standardization
scaler_cancer = StandardScaler()
X_cancer_scaled = scaler_cancer.fit_transform(X_cancer)

# Model
model_cv = LogisticRegression(max_iter=10000, random_state=42)

# 1. Single train/test split
X_train_cv, X_test_cv, y_train_cv, y_test_cv = train_test_split(
    X_cancer_scaled, y_cancer, test_size=0.2, random_state=42
)
model_cv.fit(X_train_cv, y_train_cv)
single_score = model_cv.score(X_test_cv, y_test_cv)

# 2. K-Fold Cross-Validation (K=5)
kfold_scores = cross_val_score(model_cv, X_cancer_scaled, y_cancer, cv=5)

# 3. Stratified K-Fold (Class balance)
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
stratified_scores = cross_val_score(model_cv, X_cancer_scaled, y_cancer, cv=stratified_kfold)

# Results
print("\n" + "="*70)
print("CROSS-VALIDATION NATIJALAR")
print("="*70)
print(f"\n1️⃣ Single Train/Test Split (80/20):")
print(f"   Accuracy: {single_score:.4f}")
print(f"\n2️⃣ K-Fold Cross-Validation (K=5):")
print(f"   Fold scores: {kfold_scores}")
print(f"   Mean accuracy: {kfold_scores.mean():.4f} ± {kfold_scores.std():.4f}")
print(f"\n3️⃣ Stratified K-Fold Cross-Validation (K=5):")
print(f"   Fold scores: {stratified_scores}")
print(f"   Mean accuracy: {stratified_scores.mean():.4f} ± {stratified_scores.std():.4f}")
print(f"\n💡 Cross-validation ishonchliroq - ko'p marta test qilindi!")
print("="*70)

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['Single Split', 'K-Fold', 'Stratified K-Fold']
means = [single_score, kfold_scores.mean(), stratified_scores.mean()]
stds = [0, kfold_scores.std(), stratified_scores.std()]

bars = ax.bar(methods, means, yerr=stds, capsize=10, alpha=0.7,
               color=['steelblue', 'orange', 'green'], edgecolor='black', linewidth=1.5)
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Train/Test Split vs Cross-Validation', fontsize=14, fontweight='bold')
ax.set_ylim(0.9, 1.0)
ax.grid(True, alpha=0.3, axis='y')

# Values on bars
for bar, mean, std in zip(bars, means, stds):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + std + 0.005,
            f'{mean:.4f}\n±{std:.4f}' if std > 0 else f'{mean:.4f}',
            ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

### 📊 Cross-Validation: Batafsil Tahlil

In [ ]:
# Batafsil cross-validation (multiple metrics)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

cv_results = cross_validate(
    model_cv, X_cancer_scaled, y_cancer,
    cv=5, scoring=scoring, return_train_score=True
)

# Results DataFrame
results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Train Mean': [
        cv_results['train_accuracy'].mean(),
        cv_results['train_precision'].mean(),
        cv_results['train_recall'].mean(),
        cv_results['train_f1'].mean(),
        cv_results['train_roc_auc'].mean()
    ],
    'Test Mean': [
        cv_results['test_accuracy'].mean(),
        cv_results['test_precision'].mean(),
        cv_results['test_recall'].mean(),
        cv_results['test_f1'].mean(),
        cv_results['test_roc_auc'].mean()
    ],
    'Test Std': [
        cv_results['test_accuracy'].std(),
        cv_results['test_precision'].std(),
        cv_results['test_recall'].std(),
        cv_results['test_f1'].std(),
        cv_results['test_roc_auc'].std()
    ]
})

print("\n" + "="*70)
print("CROSS-VALIDATION: MULTIPLE METRICS")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(results_df))
width = 0.35

bars1 = ax.bar(x - width/2, results_df['Train Mean'], width, label='Train',
               alpha=0.8, color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, results_df['Test Mean'], width,
               yerr=results_df['Test Std'], capsize=5, label='Test',
               alpha=0.8, color='orange', edgecolor='black')

ax.set_xlabel('Metric', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Cross-Validation: Train vs Test (Multiple Metrics)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Metric'], fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0.9, 1.01)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---

# 3️⃣ Overfitting va Underfitting

## 📖 Nazariya

**Model Complexity** va **Error** orasidagi munosabat:

### Underfitting (Underlearning)
- **Sabab**: Model juda oddiy
- **Belgilari**:
  - Train error yuqori 📈
  - Test error yuqori 📈
  - Model pattern'larni o'rganmagan
- **Yechim**:
  - Ko'proq feature qo'shish
  - Murakkab model tanlash
  - Ko'proq train qilish

### Good Fit (Optimal)
- **Sabab**: Model to'g'ri tanlan gan
- **Belgilari**:
  - Train error past ✅
  - Test error past ✅
  - Train va Test error yaqin
- **Natija**: Production'ga tayyor!

### Overfitting (Overlearning)
- **Sabab**: Model juda murakkab
- **Belgilari**:
  - Train error juda past 📉
  - Test error yuqori 📈
  - **Gap** katta!
- **Yechim**:
  - Ko'proq training data
  - Regularization (L1, L2)
  - Feature selection
  - Ensemble methods
  - Early stopping
  - Dropout (Neural Networks)

### Bias-Variance Trade-off

| **Xususiyat** | **High Bias (Underfitting)** | **High Variance (Overfitting)** |
|---------------|------------------------------|--------------------------------|
| **Model** | Juda oddiy | Juda murakkab |
| **Train Error** | Yuqori | Past |
| **Test Error** | Yuqori | Yuqori |
| **Generalization** | Yomon | Yomon |
| **Example** | Linear model (non-linear data) | Deep tree (small data) |

---

## 🎨 Overfitting vs Underfitting: Vizual Demo

In [ ]:
# Non-linear data yaratish
np.random.seed(42)
n = 100
X_fit = np.linspace(0, 10, n).reshape(-1, 1)
y_fit = 2 * np.sin(X_fit).ravel() + np.random.normal(0, 0.5, n)

# Train/Test split
X_train_fit, X_test_fit, y_train_fit, y_test_fit = train_test_split(
    X_fit, y_fit, test_size=0.3, random_state=42
)

# 3 xil model: Underfitting, Good Fit, Overfitting
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Model 1: Underfitting (degree=1, linear)
model_underfit = make_pipeline(PolynomialFeatures(degree=1), Ridge())
model_underfit.fit(X_train_fit, y_train_fit)

# Model 2: Good Fit (degree=3)
model_goodfit = make_pipeline(PolynomialFeatures(degree=3), Ridge())
model_goodfit.fit(X_train_fit, y_train_fit)

# Model 3: Overfitting (degree=15)
model_overfit = make_pipeline(PolynomialFeatures(degree=15), Ridge(alpha=0.00001))
model_overfit.fit(X_train_fit, y_train_fit)

# Predictions
X_plot = np.linspace(0, 10, 300).reshape(-1, 1)
y_underfit = model_underfit.predict(X_plot)
y_goodfit = model_goodfit.predict(X_plot)
y_overfit = model_overfit.predict(X_plot)

# Scores
from sklearn.metrics import mean_squared_error, r2_score

models_fit = [
    ('Underfitting', model_underfit),
    ('Good Fit', model_goodfit),
    ('Overfitting', model_overfit)
]

scores_fit = []
for name, model in models_fit:
    train_score = r2_score(y_train_fit, model.predict(X_train_fit))
    test_score = r2_score(y_test_fit, model.predict(X_test_fit))
    train_mse = mean_squared_error(y_train_fit, model.predict(X_train_fit))
    test_mse = mean_squared_error(y_test_fit, model.predict(X_test_fit))
    scores_fit.append({
        'Model': name,
        'Train R²': train_score,
        'Test R²': test_score,
        'Train MSE': train_mse,
        'Test MSE': test_mse
    })

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: Model fits
titles = ['Underfitting (Degree=1)', 'Good Fit (Degree=3)', 'Overfitting (Degree=15)']
predictions = [y_underfit, y_goodfit, y_overfit]

for idx, (title, y_pred) in enumerate(zip(titles, predictions)):
    axes[0, idx].scatter(X_train_fit, y_train_fit, s=40, alpha=0.7, 
                         c='blue', edgecolors='k', label='Train Data')
    axes[0, idx].scatter(X_test_fit, y_test_fit, s=40, alpha=0.7,
                         c='red', edgecolors='k', label='Test Data')
    axes[0, idx].plot(X_plot, y_pred, 'g-', linewidth=2, label='Model')
    axes[0, idx].set_xlabel('X', fontsize=11, fontweight='bold')
    axes[0, idx].set_ylabel('y', fontsize=11, fontweight='bold')
    axes[0, idx].set_title(title, fontsize=13, fontweight='bold')
    axes[0, idx].legend(fontsize=9)
    axes[0, idx].grid(True, alpha=0.3)

# Row 2: Scores comparison
scores_df = pd.DataFrame(scores_fit)

# R² Score
axes[1, 0].bar(scores_df['Model'], scores_df['Train R²'], alpha=0.7,
               label='Train R²', color='steelblue', edgecolor='black', width=0.4, position=0)
axes[1, 0].bar(scores_df['Model'], scores_df['Test R²'], alpha=0.7,
               label='Test R²', color='orange', edgecolor='black', width=0.4, position=1)
axes[1, 0].set_ylabel('R² Score', fontsize=11, fontweight='bold')
axes[1, 0].set_title('R² Score Comparison', fontsize=13, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3, axis='y')
axes[1, 0].set_xticklabels(scores_df['Model'], rotation=15, ha='right')

# MSE
axes[1, 1].bar(scores_df['Model'], scores_df['Train MSE'], alpha=0.7,
               label='Train MSE', color='steelblue', edgecolor='black', width=0.4, position=0)
axes[1, 1].bar(scores_df['Model'], scores_df['Test MSE'], alpha=0.7,
               label='Test MSE', color='orange', edgecolor='black', width=0.4, position=1)
axes[1, 1].set_ylabel('MSE', fontsize=11, fontweight='bold')
axes[1, 1].set_title('MSE Comparison', fontsize=13, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3, axis='y')
axes[1, 1].set_xticklabels(scores_df['Model'], rotation=15, ha='right')

# Gap (Train - Test)
axes[1, 2].bar(scores_df['Model'], 
               scores_df['Train R²'] - scores_df['Test R²'],
               alpha=0.7, color=['red', 'green', 'red'], edgecolor='black')
axes[1, 2].set_ylabel('Gap (Train R² - Test R²)', fontsize=11, fontweight='bold')
axes[1, 2].set_title('Overfitting Gap', fontsize=13, fontweight='bold')
axes[1, 2].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1, 2].grid(True, alpha=0.3, axis='y')
axes[1, 2].set_xticklabels(scores_df['Model'], rotation=15, ha='right')

plt.suptitle('Underfitting vs Good Fit vs Overfitting', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Print scores
print("\n" + "="*80)
print("OVERFITTING vs UNDERFITTING - SCORES")
print("="*80)
print(scores_df.to_string(index=False))
print("\n" + "="*80)
print("TUSHUNTIRISH:")
print("="*80)
print("\n📉 Underfitting (Degree=1):")
print("   - Train R² past, Test R² past")
print("   - Model juda oddiy, pattern'ni o'rganmagan")
print("\n✅ Good Fit (Degree=3):")
print("   - Train R² yaxshi, Test R² yaxshi")
print("   - Train va Test yaqin - generalize qiladi!")
print("\n📈 Overfitting (Degree=15):")
print("   - Train R² juda yuqori, lekin Test R² past")
print("   - Gap katta - training data'ga juda moslashgan!")
print("="*80)

---

# 4️⃣ Classification Metrics

## 📖 Nazariya

Classification task'larda model qanchalik yaxshi ishlashini baholash uchun turli metriclar mavjud.

### Confusion Matrix (Chalkashlik Matri tsasi)

```
                    Predicted
                Positive  Negative
Actual Positive    TP        FN
       Negative    FP        TN
```

- **TP (True Positive)**: To'g'ri Positive - kasalni kasal deb aniqladi ✅
- **TN (True Negative)**: To'g'ri Negative - sog'lomni sog'lom deb aniqladi ✅
- **FP (False Positive)**: Noto'g'ri Positive - sog'lomni kasal deb aniqladi ❌ (Type I Error)
- **FN (False Negative)**: Noto'g'ri Negative - kasalni sog'lom deb aniqladi ❌ (Type II Error)

### Metrics Formulalari

#### 1. Accuracy (To'g'rilik)
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

**Ma'nosi**: Umumiy to'g'rilik  
**Qachon**: Balanced dataset  
**Kamchilik**: Imbalanced data'da misleading

#### 2. Precision (Aniqlik)
$$\text{Precision} = \frac{TP}{TP + FP}$$

**Ma'nosi**: Positive deb aylanganlardan qanchasi haqiqatan positive?  
**Qachon**: False Positive xavfli (spam detection)  
**Misol**: "Spam" deb belgilan gan emaillardan qanchasi haqiqatan spam?

#### 3. Recall (Sezgirlik, Sensitivity, TPR)
$$\text{Recall} = \frac{TP}{TP + FN}$$

**Ma'nosi**: Barcha positive'lardan qanchasini topdik?  
**Qachon**: False Negative xavfli (medical diagnosis)  
**Misol**: Barcha kasallardan qanchasini aniqladik?

#### 4. F1-Score (Harmonik O'rtacha)
$$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Ma'nosi**: Precision va Recall'ning muvozanati  
**Qachon**: Imbalanced data, ikkalasi ham muhim  
**Range**: 0-1 (1 = best)

#### 5. Specificity (True Negative Rate)
$$\text{Specificity} = \frac{TN}{TN + FP}$$

**Ma'nosi**: Barcha negative'lardan qanchasini to'g'ri aniqladik?  
**Qachon**: Negative class muhim

---

### Qaysi Metric Qachon?

| **Vazifa** | **Muhim Metric** | **Sabab** |
|------------|------------------|-----------|
| **Medical Diagnosis** | **Recall** ↑ | Kasalni o'tkazib yubormaslik |
| **Spam Detection** | **Precision** ↑ | Yaxshi emailni spam qilmaslik |
| **Fraud Detection** | **F1-Score** | Ikkalasi ham muhim |
| **Balanced Dataset** | **Accuracy** | Oddiy va yetarli |
| **Imbalanced Dataset** | **F1, ROC-AUC** | Accuracy misleading |

---

## 🎨 Confusion Matrix: Vizual Demo

In [ ]:
# Breast Cancer dataset
X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_cancer_scaled, y_cancer, test_size=0.2, random_state=42
)

# Model
model_bc = LogisticRegression(max_iter=10000, random_state=42)
model_bc.fit(X_train_bc, y_train_bc)

# Predictions
y_pred_bc = model_bc.predict(X_test_bc)

# Confusion Matrix
cm = confusion_matrix(y_test_bc, y_pred_bc)

# Metrics
accuracy = accuracy_score(y_test_bc, y_pred_bc)
precision = precision_score(y_test_bc, y_pred_bc)
recall = recall_score(y_test_bc, y_pred_bc)
f1 = f1_score(y_test_bc, y_pred_bc)

# Calculate TP, TN, FP, FN
tn, fp, fn, tp = cm.ravel()

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            square=True, linewidths=2, linecolor='black',
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'],
            annot_kws={'size': 16, 'weight': 'bold'})
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=12, fontweight='bold')

# Add TP, TN, FP, FN labels
axes[0].text(0.5, 0.25, f'TN={tn}', ha='center', va='center', fontsize=12, color='darkblue')
axes[0].text(1.5, 0.25, f'FP={fp}', ha='center', va='center', fontsize=12, color='darkred')
axes[0].text(0.5, 1.25, f'FN={fn}', ha='center', va='center', fontsize=12, color='darkred')
axes[0].text(1.5, 1.25, f'TP={tp}', ha='center', va='center', fontsize=12, color='darkgreen')

# Metrics Bar Chart
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrics_values = [accuracy, precision, recall, f1]
colors = ['steelblue', 'orange', 'green', 'red']

bars = axes[1].barh(metrics_names, metrics_values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_xlabel('Score', fontsize=12, fontweight='bold')
axes[1].set_title('Classification Metrics', fontsize=14, fontweight='bold')
axes[1].set_xlim(0, 1)
axes[1].grid(True, alpha=0.3, axis='x')

# Values on bars
for bar, value in zip(bars, metrics_values):
    axes[1].text(value + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{value:.4f}', va='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

# Print results
print("\n" + "="*70)
print("CONFUSION MATRIX & METRICS")
print("="*70)
print("\nConfusion Matrix:")
print(f"  TN (True Negative):  {tn:3d}  |  FP (False Positive): {fp:3d}")
print(f"  FN (False Negative): {fn:3d}  |  TP (True Positive):  {tp:3d}")
print("\nMetrics:")
print(f"  Accuracy:  {accuracy:.4f} = (TP+TN)/(TP+TN+FP+FN) = ({tp}+{tn})/({tp}+{tn}+{fp}+{fn})")
print(f"  Precision: {precision:.4f} = TP/(TP+FP) = {tp}/({tp}+{fp})")
print(f"  Recall:    {recall:.4f} = TP/(TP+FN) = {tp}/({tp}+{fn})")
print(f"  F1-Score:  {f1:.4f} = 2 * (Precision * Recall) / (Precision + Recall)")
print("\n" + "="*70)
print("TUSHUNTIRISH:")
print("="*70)
print(f"  ✅ {tp} kasallarni to'g'ri aniqladik (TP)")
print(f"  ✅ {tn} sog'lomlarni to'g'ri aniqladik (TN)")
print(f"  ❌ {fp} sog'lomni kasal deb xato aniqladik (FP) - False Alarm")
print(f"  ❌ {fn} kasalni sog'lom deb o'tkazib yubordik (FN) - Xavfli!")
print("="*70)

## 📊 Classification Report

In [ ]:
# Classification Report - barcha metriclar bitta joyda
print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test_bc, y_pred_bc, target_names=['Malignant (0)', 'Benign (1)']))
print("="*70)
print("\n💡 Classification Report Tushuntirish:")
print("  - precision: Positive deb aytganlardan qanchasi to'g'ri")
print("  - recall: Barcha positive'lardan qanchasini topdik")
print("  - f1-score: Precision va Recall'ning harmonik o'rtachasi")
print("  - support: Har bir class'da nechta sample bor")
print("  - macro avg: Har bir class'ning oddiy o'rtachasi")
print("  - weighted avg: Class balance hisobga olingan o'rtacha")
print("="*70)

---

# 5️⃣ ROC Curve va AUC

## 📖 Nazariya

**ROC (Receiver Operating Characteristic) Curve** - classification model'ning threshold'ga bog'liq holda ishlashini ko'rsatadi.

### Asosiy Kontseptsiyalar

#### TPR (True Positive Rate) = Recall = Sensitivity
$$TPR = \frac{TP}{TP + FN}$$

Barcha positive'lardan qanchasini topdik?

#### FPR (False Positive Rate)
$$FPR = \frac{FP}{FP + TN}$$

Barcha negative'lardan qanchasini noto'g'ri positive deb ayitdik?

### ROC Curve
- **X o'qi**: FPR (False Positive Rate)
- **Y o'qi**: TPR (True Positive Rate / Recall)
- **Har bir nuqta**: Turli threshold qiymati

### AUC (Area Under Curve)
- **Range**: 0 - 1
- **Ma'nosi**: ROC curve ostidagi maydon
- **Interpretation**:
  - **AUC = 1.0**: Perfect classifier ⭐
  - **AUC = 0.9-1.0**: Excellent 🌟
  - **AUC = 0.8-0.9**: Good ✅
  - **AUC = 0.7-0.8**: Fair ⚖️
  - **AUC = 0.6-0.7**: Poor ⚠️
  - **AUC = 0.5**: Random guess (diagonal line) 🎲
  - **AUC < 0.5**: Worse than random ❌

### Threshold Tuning

Classifier odatda **probability** qaytaradi (0-1). **Threshold** - positive deb belgilash uchun minimal probability.

**Default**: threshold = 0.5
- P(y=1) >= 0.5 → Positive
- P(y=1) < 0.5 → Negative

**Threshold ni o'zgartirish**:
- **Threshold ↑** (masalan, 0.8): Precision ↑, Recall ↓
- **Threshold ↓** (masalan, 0.3): Precision ↓, Recall ↑

### Qaysi Threshold Tanlash?

| **Vazifa** | **Threshold** | **Sabab** |
|------------|---------------|-----------|
| **Medical Diagnosis** | Past (0.3) | Recall ↑ - kasalni o'tkazib yubormaslik |
| **Spam Detection** | Yuqori (0.7) | Precision ↑ - yaxshi emailni spam qilmaslik |
| **Balanced** | O'rtacha (0.5) | Precision va Recall balans |

---

## 🎨 ROC Curve: Vizual Demo

In [ ]:
# Probability predictions
y_proba_bc = model_bc.predict_proba(X_test_bc)[:, 1]  # Probability of positive class

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test_bc, y_proba_bc)
roc_auc = roc_auc_score(y_test_bc, y_proba_bc)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curve
axes[0].plot(fpr, tpr, color='darkorange', lw=3, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier (AUC = 0.5)')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate (FPR)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Positive Rate (TPR / Recall)', fontsize=12, fontweight='bold')
axes[0].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[0].legend(loc="lower right", fontsize=11)
axes[0].grid(True, alpha=0.3)

# Threshold effect
# Test different thresholds
thresholds_test = [0.3, 0.5, 0.7, 0.9]
metrics_threshold = []

for thresh in thresholds_test:
    y_pred_thresh = (y_proba_bc >= thresh).astype(int)
    prec = precision_score(y_test_bc, y_pred_thresh)
    rec = recall_score(y_test_bc, y_pred_thresh)
    f1_thresh = f1_score(y_test_bc, y_pred_thresh)
    metrics_threshold.append({'Threshold': thresh, 'Precision': prec, 'Recall': rec, 'F1': f1_thresh})

df_thresh = pd.DataFrame(metrics_threshold)

# Plot threshold effect
axes[1].plot(df_thresh['Threshold'], df_thresh['Precision'], 'o-', linewidth=2, markersize=8, label='Precision')
axes[1].plot(df_thresh['Threshold'], df_thresh['Recall'], 's-', linewidth=2, markersize=8, label='Recall')
axes[1].plot(df_thresh['Threshold'], df_thresh['F1'], '^-', linewidth=2, markersize=8, label='F1-Score')
axes[1].set_xlabel('Threshold', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[1].set_title('Threshold Effect on Metrics', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.8, 1.0)

plt.tight_layout()
plt.show()

# Print results
print("\n" + "="*70)
print("ROC CURVE & AUC")
print("="*70)
print(f"\nAUC Score: {roc_auc:.4f}")
if roc_auc >= 0.9:
    print("  Interpretation: Excellent model! 🌟")
elif roc_auc >= 0.8:
    print("  Interpretation: Good model! ✅")
elif roc_auc >= 0.7:
    print("  Interpretation: Fair model ⚖️")
else:
    print("  Interpretation: Poor model ⚠️")

print("\n" + "="*70)
print("THRESHOLD TUNING")
print("="*70)
print(df_thresh.to_string(index=False))
print("\n💡 Kuzatish:")
print("  - Threshold ↑ → Precision ↑, Recall ↓")
print("  - Threshold ↓ → Precision ↓, Recall ↑")
print("  - Optimal threshold vazifaga bog'liq!")
print("="*70)

## 🔄 Multiple Models Comparison

In [ ]:
# Ko'p modellarni taqqoslash
models = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

# Train va evaluate
results_models = []
roc_curves = {}

for name, model in models.items():
    # Train
    model.fit(X_train_bc, y_train_bc)
    
    # Predict
    y_pred = model.predict(X_test_bc)
    y_proba = model.predict_proba(X_test_bc)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_test_bc, y_pred)
    prec = precision_score(y_test_bc, y_pred)
    rec = recall_score(y_test_bc, y_pred)
    f1 = f1_score(y_test_bc, y_pred)
    auc_score = roc_auc_score(y_test_bc, y_proba)
    
    # ROC Curve
    fpr_model, tpr_model, _ = roc_curve(y_test_bc, y_proba)
    roc_curves[name] = (fpr_model, tpr_model)
    
    # Save results
    results_models.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc_score
    })

# Results DataFrame
df_results = pd.DataFrame(results_models)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Metrics Comparison
df_plot = df_results.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]
df_plot.plot(kind='bar', ax=axes[0], alpha=0.8, edgecolor='black', linewidth=1.2)
axes[0].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[0].set_title('Models Comparison: All Metrics', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].set_ylim(0.9, 1.0)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_xticklabels(df_results['Model'], rotation=45, ha='right')

# ROC Curves Comparison
colors = ['blue', 'green', 'red', 'purple']
for (name, (fpr_m, tpr_m)), color in zip(roc_curves.items(), colors):
    auc_val = df_results[df_results['Model'] == name]['ROC-AUC'].values[0]
    axes[1].plot(fpr_m, tpr_m, lw=2, label=f'{name} (AUC={auc_val:.3f})', color=color)

axes[1].plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC=0.5)')
axes[1].set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
axes[1].set_title('ROC Curves Comparison', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print results
print("\n" + "="*80)
print("MULTIPLE MODELS COMPARISON")
print("="*80)
print(df_results.to_string(index=False))
print("\n" + "="*80)
print("BEST MODEL PER METRIC:")
print("="*80)
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    best_idx = df_results[metric].idxmax()
    best_model = df_results.loc[best_idx, 'Model']
    best_value = df_results.loc[best_idx, metric]
    print(f"  {metric:12s}: {best_model:20s} ({best_value:.4f})")
print("="*80)

## 📊 Regression Metrics

Regression muammolarida model baholash uchun boshqa metrikalar qo'llaniladi:

### 1. **MAE (Mean Absolute Error)** - O'rtacha Absolyut Xato
$$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

**Xususiyatlari:**
- Barcha xatolar bir xil og'irlikka ega
- Tushunish oson
- Outliers ta'siri kam

---

### 2. **MSE (Mean Squared Error)** - O'rtacha Kvadrat Xato
$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Xususiyatlari:**
- Katta xatolar ko'proq jazalanadi
- Outliers ta'siri yuqori
- Differensiallanuvchi (optimizatsiya uchun qulay)

---

### 3. **RMSE (Root Mean Squared Error)** - Ildiz O'rtacha Kvadrat Xato
$$RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

**Xususiyatlari:**
- MSE ning ildizi (y bilan bir xil birlikda)
- Interpretatsiya qilish oson
- Eng ko'p qo'llaniladigan metrika

---

### 4. **R² (Coefficient of Determination)** - Determinatsiya Koeffitsienti
$$R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$$

**Interpretatsiya:**
- **R² = 1**: Perfect model (100% variance explained)
- **R² = 0**: Model o'rtacha qiymatdan yaxshiroq emas
- **R² < 0**: Model o'rtacha qiymatdan ham yomon

| R² Score | Model Sifati |
|----------|--------------|
| 0.9 - 1.0 | A'lo (Excellent) |
| 0.8 - 0.9 | Juda yaxshi (Very Good) |
| 0.7 - 0.8 | Yaxshi (Good) |
| 0.6 - 0.7 | O'rtacha (Fair) |
| < 0.6 | Yomon (Poor) |

### 💻 Regression Metrics: Vizual Demo

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# California Housing dataset
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
X_housing = pd.DataFrame(housing.data, columns=housing.feature_names)
y_housing = housing.target

# Train-test split
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=42
)

# Standardization
scaler_h = StandardScaler()
X_train_h_scaled = scaler_h.fit_transform(X_train_h)
X_test_h_scaled = scaler_h.transform(X_test_h)

# Train multiple regression models
models_reg = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
}

results_reg = []
predictions_dict = {}

for name, model in models_reg.items():
    # Train
    model.fit(X_train_h_scaled, y_train_h)
    
    # Predict
    y_pred_train = model.predict(X_train_h_scaled)
    y_pred_test = model.predict(X_test_h_scaled)
    
    # Calculate metrics for test set
    mae = mean_absolute_error(y_test_h, y_pred_test)
    mse = mean_squared_error(y_test_h, y_pred_test)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_h, y_pred_test)
    
    # Calculate train R² to check overfitting
    r2_train = r2_score(y_train_h, y_pred_train)
    
    predictions_dict[name] = y_pred_test
    
    results_reg.append({
        'Model': name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R² (Test)': r2,
        'R² (Train)': r2_train,
        'Overfit Gap': r2_train - r2
    })

df_reg_results = pd.DataFrame(results_reg)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# 1. Metrics Comparison (Bar)
ax1 = axes[0, 0]
df_metrics = df_reg_results[['Model', 'MAE', 'RMSE']].set_index('Model')
df_metrics.plot(kind='bar', ax=ax1, alpha=0.8, edgecolor='black', linewidth=1.2, color=['#3498db', '#e74c3c'])
ax1.set_ylabel('Error', fontsize=12, fontweight='bold')
ax1.set_title('Regression Error Metrics (MAE vs RMSE)', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_xticklabels(df_reg_results['Model'], rotation=45, ha='right')

# 2. R² Score Comparison
ax2 = axes[0, 1]
x_pos = np.arange(len(df_reg_results))
width = 0.35
ax2.bar(x_pos - width/2, df_reg_results['R² (Train)'], width, 
        label='Train R²', alpha=0.8, edgecolor='black', linewidth=1.2, color='#2ecc71')
ax2.bar(x_pos + width/2, df_reg_results['R² (Test)'], width, 
        label='Test R²', alpha=0.8, edgecolor='black', linewidth=1.2, color='#e67e22')
ax2.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax2.set_title('R² Score: Train vs Test (Overfitting Check)', fontsize=14, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(df_reg_results['Model'], rotation=45, ha='right')
ax2.legend(loc='lower right', fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=0.8, color='red', linestyle='--', lw=1.5, label='Good Threshold (0.8)')

# 3. Actual vs Predicted (Linear Regression)
ax3 = axes[1, 0]
y_pred_lr = predictions_dict['Linear Regression']
ax3.scatter(y_test_h, y_pred_lr, alpha=0.5, s=30, edgecolor='black', linewidth=0.5)
ax3.plot([y_test_h.min(), y_test_h.max()], [y_test_h.min(), y_test_h.max()], 
         'r--', lw=2, label='Perfect Prediction')
ax3.set_xlabel('Actual Price', fontsize=12, fontweight='bold')
ax3.set_ylabel('Predicted Price', fontsize=12, fontweight='bold')
ax3.set_title('Linear Regression: Actual vs Predicted', fontsize=14, fontweight='bold')
ax3.legend(loc='upper left', fontsize=10)
ax3.grid(True, alpha=0.3)

# 4. Residual Plot (Random Forest)
ax4 = axes[1, 1]
y_pred_rf = predictions_dict['Random Forest']
residuals = y_test_h - y_pred_rf
ax4.scatter(y_pred_rf, residuals, alpha=0.5, s=30, edgecolor='black', linewidth=0.5)
ax4.axhline(y=0, color='red', linestyle='--', lw=2)
ax4.set_xlabel('Predicted Price', fontsize=12, fontweight='bold')
ax4.set_ylabel('Residuals (Actual - Predicted)', fontsize=12, fontweight='bold')
ax4.set_title('Random Forest: Residual Plot', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print results
print("\n" + "="*100)
print("REGRESSION METRICS COMPARISON")
print("="*100)
print(df_reg_results.to_string(index=False))
print("\n" + "="*100)
print("INTERPRETATION:")
print("="*100)
print("✅ MAE: Qanchalik kam bo'lsa, model shunchalik yaxshi")
print("✅ RMSE: MAE dan biroz katta (katta xatolarni ko'proq jazalaydi)")
print("✅ R²: 1 ga yaqin bo'lsa yaxshi (0.8+ = yaxshi model)")
print("✅ Overfit Gap: Kichik bo'lishi kerak (< 0.05 ideal)")
print("="*100)

# Best model
best_r2_idx = df_reg_results['R² (Test)'].idxmax()
best_model = df_reg_results.loc[best_r2_idx, 'Model']
best_r2 = df_reg_results.loc[best_r2_idx, 'R² (Test)']
print(f"\n🏆 BEST MODEL: {best_model} (R² = {best_r2:.4f})")
print("="*100)

## 🎯 Best Practices va Xulosa

### 📌 **Qaysi Metrikani Qachon Ishlatish Kerak?**

#### **Classification:**

| Vaziyat | Metrika | Sabab |
|---------|---------|-------|
| **Balanced Dataset** | Accuracy | Barcha sinflar teng muhim |
| **Imbalanced Dataset** | F1-Score / ROC-AUC | Minority class muhim |
| **Medical Diagnosis** | Recall | False Negative juda xavfli (kasalni o'tkazib yuborish) |
| **Spam Detection** | Precision | False Positive muammo (normal xat spam deb belgilanadi) |
| **Multi-Class** | Macro/Weighted F1 | Barcha sinflarni hisobga olish |
| **Model Selection** | ROC-AUC | Threshold dan mustaqil baholash |

---

#### **Regression:**

| Vaziyat | Metrika | Sabab |
|---------|---------|-------|
| **Interpretability** | MAE | Tushunish oson, outlier ta'siri kam |
| **Optimization** | MSE | Differensiallanuvchi, optimization uchun qulay |
| **Standard Reporting** | RMSE | Y bilan bir xil birlikda |
| **Model Comparison** | R² | Percentage variance explained (0-100%) |
| **Outliers Present** | MAE > RMSE | RMSE outliers ta'siriga sezgir |

---

### ⚠️ **Keng Tarqalgan Xatolar:**

1. **❌ Imbalanced Dataset da Accuracy ishlatish**
   - ✅ F1-Score yoki ROC-AUC ishlatish

2. **❌ Test Set ga Fit qilish (Data Leakage)**
   - ✅ Faqat Train Set ga fit qilish, Test Set faqat baholash uchun

3. **❌ Cross-Validation siz model tanlash**
   - ✅ Har doim CV bilan modelni baholash

4. **❌ Faqat bitta metrikaga qarash**
   - ✅ Bir nechta metrikalarni taqqoslash

5. **❌ Overfitting ni e'tiborsiz qoldirish**
   - ✅ Train vs Test performance ni doim tekshirish

---

### 🔑 **Model Evaluation Checklist:**

- [ ] **1. Train-Test Split** to'g'ri bajarilganmi? (80-20 yoki 70-30)
- [ ] **2. Data Leakage** yo'qmi? (Standardization, Feature Engineering)
- [ ] **3. Cross-Validation** qo'llanilganmi? (K-Fold yoki Stratified)
- [ ] **4. Overfitting** tekshirilganmi? (Train vs Test metrics)
- [ ] **5. Multiple Metrics** taqqoslanganmi? (Accuracy, Precision, Recall, F1, AUC)
- [ ] **6. Confusion Matrix** ko'rilganmi? (FP, FN tahlili)
- [ ] **7. ROC Curve** chizilganmi? (Threshold tuning)
- [ ] **8. Best Model** tanlanganmi? (Vazifaga mos metrikaga asosan)

---

### 📚 **Keyingi Qadamlar:**

1. **Practical.ipynb** - Real dataset bilan amaliy mashg'ulot
2. **Homework.md** - Mustaqil ishlash uchun vazifalar
3. **Evaluation Guide** - Tez yordam qo'llanmasi

---

### 🎓 **Xulosa:**

Model baholash - Machine Learning dagi eng muhim bosqichlardan biri. To'g'ri metrikalarni tanlash va modelni to'g'ri baholash orqali:
- ✅ Real worldda yaxshi ishlash
- ✅ Overfitting ni oldini olish
- ✅ Ishonchli natijalar olish

**Esda tuting:** *"A model is only as good as how well you evaluate it!"* 🚀